# RAG

In [1]:
# Load the environment
from dotenv import load_dotenv
load_dotenv()

True

In [2]:
# Read the model name
import os
MODEL_NAME = os.environ["GEMINI_MODEL"]
API_KEY = os.environ["GOOGLE_GENERATIVE_AI_API_KEY"]

In [3]:
# Create the LangChain model
from langchain_google_genai import ChatGoogleGenerativeAI
llm = ChatGoogleGenerativeAI (model = MODEL_NAME, api_key =API_KEY, temperature = 0)

In [4]:
# Create the document object
from langchain_core.documents import Document
documents = [
    Document(
        page_content="LangChain provides abstractions for building LLM applications.",
        metadata={"source": "langchain.txt"}
    ),
    Document(
        page_content="LangGraph is designed for stateful agent workflows.",
        metadata={"source": "langgraph.txt"}
    ),
]

## Manual RAG

In [5]:
# Create the embedding model
from langchain_google_genai import GoogleGenerativeAIEmbeddings

embeddings = GoogleGenerativeAIEmbeddings(
    model="models/gemini-embedding-001",
    google_api_key=API_KEY,
)

In [6]:
# Create the vector store
from langchain_core.vectorstores import InMemoryVectorStore

vector_store = InMemoryVectorStore(embedding=embeddings)

In [7]:
# Add documents to the vector_store

vector_store.add_documents(documents)

['2e340926-7954-4748-b70b-cf4100f33991',
 '6d1da7bc-0f76-48ac-bb71-5fcf82957d95']

In [8]:
# Run the retriever

retriever = vector_store.as_retriever(
    search_kwargs = {"k":2}
)

In [9]:
# Invoke the retriever

query = "What is LangGraph"
retrieved_docs = retriever.invoke(query)

In [10]:
# Create the formatting function
def format_docs(documents: list[Document])->str:
    return "\n\n".join(
        doc.page_content
        for doc in documents
    )

"""
or
def docs_to_text(documents: list[Document])-> str:
    text = ""
    for doc in documents:
        text = text + doc.page_content + "\n\n"
    return text
"""

'\nor\ndef docs_to_text(documents: list[Document])-> str:\n    text = ""\n    for doc in documents:\n        text = text + doc.page_content + "\n\n"\n    return text\n'

In [11]:
context = format_docs(retrieved_docs)
print(context)

LangGraph is designed for stateful agent workflows.

LangChain provides abstractions for building LLM applications.


In [12]:
# Create the prompt

prompt = f"""
Answer the question using the following context.

Context:
{context}

Question:
{query}
"""

In [13]:
# Invoke the llm

response = llm.invoke(prompt)

print(response.content)

[{'type': 'text', 'text': 'Based on the provided context, LangGraph is designed for stateful agent workflows.', 'extras': {'signature': 'EoAGCv0FARFNMg/IwlUwkq+YbSM/Gq+hMqYjovHW0OWAUhuthbJDV8Jh6GzRqq0NyOZmQc6bIP4gOem0qHSPLt1zio/ji5dqrlPOudG3/270e4UDd5mZn340gyWHPg/4fXq9CC0o9NwV/IcKQ8m3KHrksWGDdZ5KJSKpRHNkm/kDjphdKeojWqMhhmM+tzsO4hGZFr6W6DdGNiL0EaW//dWJ2LVxP87JsCj8fqg7/Vb4sY2MBUtq8Sf7zMEIlEWYQCyyAoqMGt4CbU0BU1kdzXWo8F1lgEpapW0dC+qfZNVlOwvAv8iis+eD4Q2z6Y1vzHU7T6sWf/+tK51OSi1r3Rm3ffgVeCd4G3hTlE0aJ3EAhWE5TrcubeHgmYPaAX/IY+zVfjID2K6Nd1JktDFr7zaKil02z/D1DC+mCmcQLtSYL4zgM5qq4QW+nZopNgEP7b4CEVMxvU3jjZcyoP0W0BDfZT+FqdcwCscq6WFwhDaTOAJ45TT1O3dBMoza4rpqkbODxq9cdyOAF6/eW71Iw+QcGy52sPK1ac6lKXU3ePEeScXOx0TooT/AKPS5WWsBywEoKpXX9iQN4NGxzZIKg40B7MH5PSQxyRwNznxYgHoJJBoBUdchZW1nHw9X9iNKao/y42hvlII+EhZR0VORGpUy1YC+oER0lWQKh8oQCiJO/B/tOdLDIPXQU71palDMTLS+MDK8MLFIEFWMHhGl14lG7K8P3a0kfBsrTCOusJC8X5GLqHbvT6qg9ePo03TKHTe/qE7F9uNHHfZwp7CRbiSfXgaSXoMAL5+BQLNNKA/uR754lnYR6bWdFm8wAsrW+BGJG1jvrpuw9AF7YjQ65KTkPaAQ3Ku

# RAG Chain with Runnables

In [14]:
query = "What is LangGraph?"

In [15]:
# Convert format_docs, prompt to runnable
from langchain_core.runnables import RunnableLambda

format_docs_runnable = RunnableLambda(format_docs)

In [16]:
from langchain_core.runnables import RunnableParallel, RunnablePassthrough

retriever_chain = RunnableParallel(
    {
    "context": retriever | format_docs_runnable,
    "query": RunnablePassthrough()
    }
)

In [17]:
# Create prompt
from langchain_core.prompts import ChatPromptTemplate


prompt = ChatPromptTemplate.from_template(
"""
Answer the question using the following context.

If the context does not contain enough information to answer the question, 
say that you don't have enough information.

Context:
{context}

Question:
{query}
""")

In [18]:
# Debug 
retrieved = retriever_chain.invoke(query)
print(retrieved['context'])

LangGraph is designed for stateful agent workflows.

LangChain provides abstractions for building LLM applications.


In [19]:
# Create RAG Chain
RAG_Chain = retriever_chain | prompt | llm

In [20]:
RAG_response = RAG_Chain.invoke(query)

In [21]:
print(RAG_response.content)

[{'type': 'text', 'text': 'Based on the provided context, LangGraph is designed for stateful agent workflows.', 'extras': {'signature': 'EskICsYIARFNMg8zw9Uif5Vsrz7KwWhItufO2HprfEDTR5OEww7gc+EUgscPhoLMrcjV1aRsQX73tuqY0jRvywhaHz5oLiZ760Ua1hRCdK9jkpdU6muAy2mIZ0zia7+rKs7KSdLs1PFYOtxFbv2ga8OXdl7/YTvP9ojIkTg4dXP7IGGZ1vhDd6nUW/6SL7bKESlG9fbfcBtCwO3imALNIjoHV6UyrY5ZHMbdTbx17PpMKIHuOKVx3BSutXVSDlM9HSMsIDmCqajO4w+Yd5iSLoGkJXof2xmIPOD5BKn54TG94WNZVHB/s5aEY/RQSM9QIWSlqIm7IY/vcEEB4MDaeMogjjj4qAKnZNrOOSheFxehJ1Dnx1DymbggZLiZPwXD0mfNuFRJeC6aUbd2Tu4l1s0WLv00xDotwYnFA6sT1EN35F8F4pCgZ4KShXcQuPOn8PCjpMLCHHi7V4pEHtOwZ+cZrUTcZn8amDtu+gz0VfTxS2dr42OqzNYj9m/+edBw/uA4baoGCRSVyoE7vDp0ts+kag9wdr7aZFUO450YvfvEiAkShPFA9Tt7tNFtZiYMwuua6g+SH3c2sJ/ru3tsdpwJjNFB49JRQXRTP2d10flfyntc0DtbgLnKX067/nmONfgNaxLcVm0YQt0HSWQukiQCr0NVi1tn5JI2YZ+JTC4bsPt7Qnc1Au8Wjw/pkW8CsC7bPgQdY/E0xPjazyTMLVqrNRS7kqOPqeK+p3QvfUitq2pjO/Tf/rqA431NtUUNwx8Y0yK5jp2W9mBtKFtR3oOqDNXG0cx8VR9bJteTSnbpAOPlZBf4uX1JUaTUJmV/1uluPwRNjswLJxb7Ig2Hyf7XFLgj1+t

RAG 2 major stages:

```text
                    RAG
                     │
          ┌──────────┴──────────┐
          ▼                     ▼
     Retrieval stage       Generation stage
          │                     │
     Find relevant docs    Generate answer
```

In [22]:
response = RAG_Chain.invoke("What is the newest AMD GPU chip")

In [23]:
print(response.content)

[{'type': 'text', 'text': "I don't have enough information.", 'extras': {'signature': 'EpkICpYIARFNMg+F/C4svJgvPJw/TacYZME3IjLpO6ca5Mda3c7Q/ZSYlVEtCGPV+uER6rUjyIGn4AiFm5FhgV1yms53GmbvRyztHwthQ1BnZ6zg9qsgvd7Hgg58JzTsjbdPdwRVw22p3SSoqOoIaKIk3JT0jgVRRdVHZKTFqVMB6vF5wIWxhQy7siXjBsF+/lHCP39KsmSsRPzVxYATKY9hmE6nBB+3JHlBVdgVYK3+35nf6NzFEUGWp0CJjkjsDuOFesLZ0cJX7yF32avt9hlHfkg2BHQU1mhYAbjXL33AlVjz5lY1Xj6/Mc1ILtm91rHq4YPgI9dVYoe5R8sOH4nUVASI+KOceqUvDSqtj1u6eIDM3S62yQmxhEoUwWcYXS5IEgLibelnXO8TsuQDO6O+eiwpmpadR5Hzp9IWEQwd2ztQJDi5tI51C1QesTqGVRKtMQuQgeGI0aOwnhRpYoczBP/IJQKp0o/3/b0MGnmH8dFl+h+dLm2eukdnCSJNKuui0rMvd6CLG1p4/IWh2M4HTUTCpU1jVNxSPOz46g/FbbUjq9nAggyEmkJszTmJ04kQKXnA8RY2wvnKFEOd1EyNgomWXHLiwt230UYAv7YLIb5Q5dC7DjPWWjXwVeIb3yTFLMzipV+QG61EuIjYlY1yB93jd3gF7AbQRFxTaB2n7Plph2way60U04g+cR2slcOvChN6BrfM65WZb6fEKBGxaZMocMD/YsdDBbuSvHlQU16pMiWjIH+vMD7O7GIb+SgH2l1Xy3yUde8oxulhAG05Za5Cddu1b3FAww3DfMd23JFAm7yft3XGXhQp5od6T8YOuJ4XRT6ULmQDYP2L7XAG0XGSuv6Vpxuz2ODWz3HHQbgg4eQThR6LPxCyi8imx7KxoCSV7D3/2GVW5

## Advanced RAG pipeline

Work Flow:
```text
Conversation history
        ↓
Query Rewriter
        ↓
Standalone query
        ↓
Retriever (MMR)
        ↓
Relevant documents
        ↓
Formatted context
        ↓
History + current question + context
        ↓
LLM
        ↓
Answer
```

In [24]:
# Create the document objects
from langchain_core.documents import Document

documents = [
    Document(
        page_content="LangChain provides abstractions for building LLM applications.",
        metadata={"source": "langchain.txt"}
    ),
    Document(
        page_content="LangGraph is designed for stateful agent workflows.",
        metadata={"source": "langgraph.txt"}
    ),
    Document(
        page_content="Retrievers are components that return relevant documents for a given query.",
        metadata={"source": "retrievers.txt"}
    ),
    Document(
        page_content="Vector stores are used to store and search vector representations of documents.",
        metadata={"source": "vector_stores.txt"}
    ),
    Document(
        page_content="Embeddings represent text as numerical vectors that capture semantic relationships.",
        metadata={"source": "embeddings.txt"}
    ),
    Document(
        page_content="RAG combines information retrieval with language generation to provide context to an LLM.",
        metadata={"source": "rag.txt"}
    ),
    Document(
        page_content="LCEL allows LangChain components to be composed into executable pipelines using the pipe operator.",
        metadata={"source": "lcel.txt"}
    ),
    Document(
        page_content="RunnableParallel allows multiple Runnable components to execute using the same input.",
        metadata={"source": "runnable_parallel.txt"}
    ),
    Document(
        page_content="RunnablePassthrough forwards the original input without modifying it.",
        metadata={"source": "runnable_passthrough.txt"}
    ),
    Document(
        page_content="RunnableLambda converts a Python function into a Runnable component.",
        metadata={"source": "runnable_lambda.txt"}
    ),
    Document(
        page_content="Prompt templates provide a reusable structure for constructing prompts dynamically.",
        metadata={"source": "prompt_templates.txt"}
    ),
    Document(
        page_content="Structured output allows an LLM response to follow a predefined schema.",
        metadata={"source": "structured_output.txt"}
    ),
    Document(
        page_content="Pydantic models can be used to define and validate structured data returned by an LLM.",
        metadata={"source": "pydantic.txt"}
    ),
    Document(
        page_content="Tool calling allows an LLM to request that an external function or system be used.",
        metadata={"source": "tool_calling.txt"}
    ),
    Document(
        page_content="A tool call contains the name of the requested tool and the arguments generated by the LLM.",
        metadata={"source": "tool_calls.txt"}
    ),
    Document(
        page_content="The application executes a tool after receiving a tool call from the LLM.",
        metadata={"source": "tool_execution.txt"}
    ),
    Document(
        page_content="Tool descriptions help an LLM understand when a tool should be selected and how it should be used.",
        metadata={"source": "tool_descriptions.txt"}
    ),
    Document(
        page_content="Similarity search retrieves documents whose vector representations are close to the query vector.",
        metadata={"source": "similarity_search.txt"}
    ),
    Document(
        page_content="Maximum Marginal Relevance can improve retrieval diversity by balancing relevance and similarity between retrieved documents.",
        metadata={"source": "mmr.txt"}
    ),
    Document(
        page_content="Metadata filtering allows retrieval results to be restricted using attributes associated with documents.",
        metadata={"source": "metadata_filtering.txt"}
    ),
]

In [25]:
# Add documents to the vector_store

vector_store.add_documents(documents)

['3f9253f4-c41d-4b03-b816-e2179673d9fe',
 '6fc3af09-5e7f-4a5b-9e7d-449c14185ea3',
 '10144c41-9e2b-4d62-a3fa-3ead4bc67b0e',
 '40e9b7f9-d77f-4e62-9040-a8baf46083ac',
 '38174394-d0d2-498e-9d17-2dc9382c4bb0',
 'd1663d4c-3b89-4875-b4f5-1fa46db7eff9',
 'faa32117-150b-4a5f-a797-598a437ecd91',
 '1ce083f8-05a2-43e8-ad76-31c26a6bde86',
 'dd0ba0fb-4076-432c-9ef6-3e60a57c253a',
 'b282b0e1-ffff-400e-85a3-bddaeee68c1f',
 'd25f1f2a-df58-4013-99a5-fae421a8f436',
 '3101213a-f3b3-44b1-aeeb-422bf4ccb427',
 'd39c0c86-8a9a-4deb-8d01-286f540b0419',
 'd4912c28-4d37-4ea5-b788-927ade73fb21',
 'd4ad602a-e819-48f4-b1d6-30c712a8c412',
 'eb6d8612-1455-45f6-81b4-3a5c5c92a3ab',
 '66a365c0-4389-4ac5-993a-1daac8b5981d',
 '4b619161-2546-4a4c-b609-d34e65791d19',
 '44de72c1-b7b5-46ef-82a1-2346699e1e5a',
 '3ab3d14a-bc73-4efc-831b-f0e961e41fcf']

In [26]:
# Create the retriever

retriever = vector_store.as_retriever(
    search_type ="mmr",
    search_kwargs = {
        "k":3,
        "fetch_k": 10
        }
)

In [27]:
# Create the formatting function
def format_docs(documents: list[Document])->str:
    docs = "\n\n".join(
        f"Page content: {doc.page_content}\n Page_source: {doc.metadata}"
        for doc in documents
        )
    return docs
    


In [28]:
# Convert the format function to runnable
format_docs_runnable = RunnableLambda(format_docs)

In [29]:
# Create history
from langchain_core.messages import HumanMessage, AIMessage

history = [
    HumanMessage(content=query),
    AIMessage(content=RAG_response.content[0]['text'])
]



In [30]:
print(history)

[HumanMessage(content='What is LangGraph?', additional_kwargs={}, response_metadata={}), AIMessage(content='Based on the provided context, LangGraph is designed for stateful agent workflows.', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[])]


In [31]:
# Create history
"""from langchain_core.messages import HumanMessage, AIMessage

def history (query, previous_query, previous_response):
    history = [
        HumanMessage(content=query),
        AIMessage(content=response.content[0]['text'])
    ]
    return(query, history)

"""

"from langchain_core.messages import HumanMessage, AIMessage\n\ndef history (query, previous_query, previous_response):\n    history = [\n        HumanMessage(content=query),\n        AIMessage(content=response.content[0]['text'])\n    ]\n    return(query, history)\n\n"

In [ ]:
retriever_chain = RunnableParallel({
    "context": retriever | format_docs_runnable,
    "query": RunnablePassthrough(),

})

In [116]:
query2 = "what is the use case of it?"

In [117]:
# Query rewriter 
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
query_rewrite_prompt = ChatPromptTemplate([
    (
        "system",
        "Rewrite the user's latest question into a standalone question "
        "that can be understood without the conversation history. "
        ),
        MessagesPlaceholder("history"),
        ("human", "{query}")
])

In [118]:
from langchain_core.output_parsers import StrOutputParser
#query_rewriter = query_rewrite_prompt | llm 
query_rewriter = query_rewrite_prompt | llm | StrOutputParser()

In [119]:
rewritten_query = query_rewriter.invoke({
    "history": history,
    "query": query2
})


ChatGoogleGenerativeAIError: Error calling model 'gemini-3.5-flash' (RESOURCE_EXHAUSTED): 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 20, model: gemini-3.5-flash\nPlease retry in 45.220133374s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}, {'@type': 'type.googleapis.com/google.rpc.QuotaFailure', 'violations': [{'quotaMetric': 'generativelanguage.googleapis.com/generate_content_free_tier_requests', 'quotaId': 'GenerateRequestsPerDayPerProjectPerModel-FreeT

In [ ]:
type(rewritten_query)

In [ ]:
# Debug str
del str

In [93]:
print(type(str))

<class 'type'>


In [ ]:
rewritten_query = str(rewritten_query)
"""
Problem:
The output of StrOutputParser() was a TextAccessor object. Although TextAccessor behaves like a string, 
it was not accepted by the vector_store / embedding model.

Solution:
Convert the TextAccessor explicitly into a built-in python str:

    rewritten_query = str(rewritten_query)

However, this initially caused another problem because the built-in str name had been 
rebounded to a list somewhere in the jupiter notebook.

Python's built-in str cannot spontaneously  become a list. This means that
something executed earlier in hte notebook had overwritten or shadowed the nam str.

Possible causes include:

    from something import *

where the imported module exposes a name called str,

or accidentally using str as a variable name, for example:

    for str in some_list:
        ...

After the loop, str remains bound to the last value from the list.

The fix was to restore the original built-in str name and then 
convert rewritten_query from TextAccessor into an actual str.  
"""

In [97]:
retrieved_docs = retriever_chain.invoke(rewritten_query)
print(retrieved_docs)

{'context': "Page content: LangGraph is designed for stateful agent workflows.\n Page_source: {'source': 'langgraph.txt'}\n\nPage content: Embeddings represent text as numerical vectors that capture semantic relationships.\n Page_source: {'source': 'embeddings.txt'}\n\nPage content: The application executes a tool after receiving a tool call from the LLM.\n Page_source: {'source': 'tool_execution.txt'}", 'query': 'What are the use cases of LangGraph?'}


In [39]:
# Create the prompt
prompt = ChatPromptTemplate.from_messages(
    [
        ("system", "Answer the question using the provided context. If the context does not contain enough information to answer the question, say that you don't have enough information."),
        MessagesPlaceholder("history"),
        ("human", "Question: {query}, \nContext: {context}"),
    ]
)

In [40]:
prompt__history = prompt.invoke(
    {
        "history":history,
        "context":retrieved_docs,
        "query":query2
    }
)

In [41]:
RAG_response =  llm.invoke(prompt__history)

In [42]:
print(RAG_response)

content=[{'type': 'text', 'text': 'Based on the provided context, the use case of LangGraph is for **stateful agent workflows**.', 'extras': {'signature': 'Ev4JCvsJARFNMg95aZ15nd/tP5II5i19xZYzly5UO98J5TflxPC4zeTgHMRiadyubuhkbuVUuS+3HO324BnhSmjVFsPZV6XdzTgRTL8tTUOs+7Pro5mLcVYUJAgupuV/yIG0BiX0GdXarvvAnryg4meMzrYrHjKK3o1Rp3R8n6o68kbT7mwJjAHEacbm8J7GmE4WhpOWk/JzWKpX6ikbbVT1XLiBrTJUpta+YF6vM22pum7NR455VmKLBd2gFCOI2P0zbXv7Sr0vxMu9ZK28pex7qWv2qz0uqhfZzkzrh/eWJEyZ1fMZE0UvIvFqhhNqQcQy+gSusmaAKuuTKiRGLJvzmuv059D3uDxcft02KzE8JVUg4tOiJBRDukN4U5Buca2r+YxbXzV6CwMToe/eoqFkEmE85tP+Cm2tq2sVju67l+lAaIUJ3H6Xb1S3n/xl8W4qxGzNTz4fAanaDiI+r4YsN8U7rsxtiZPofuGpszIVlBpQfBDgIivBSvxfdPXcClNjVAvyoR6Uc8m9SwT3Sl1lDUY9lxXIecAa7g2X/XzlND+kHbG5sYngpjwg2S8zXWA26BGMZmvTBTech7pSTW+mi+sK7fr4qtbxqQVfef4uQdQZmRl1C3Oz5LJQUxgNajkd/Q1Pxb0uPu3FWnnO8BK0apUWh37JRZv0Bf0u7/XoY4khjs1HH+Z9q6Re721Lzs8cqzymY2+3RyBXVSyPWdzIdg7j92Ua/1F1ZliPiVvt26ltiNqGs5Az8VF6mFIC6eSFhavypWTQqnp4qPq/wK1HSJYufh2t4SOp+f3oxs1R8QvCbfAJ7aTTIr3Cwjel/ZZIGLo6fdVN

## Advanced RAG pipeline improvement

In [43]:
query2 = "what is the use case of it?"

In [44]:
# Create the conversation input
# Wont be used any further but for illustration 
conversation_input = RunnableParallel({
    "history": RunnablePassthrough(),
    "query": RunnablePassthrough(),
})

In [45]:
# Invoke the conversation input
conversation_input.invoke({
    "history": history,
    "query": query2
}) 

{'history': {'history': [HumanMessage(content='What is LangGraph?', additional_kwargs={}, response_metadata={}),
   AIMessage(content='Based on the provided context, LangGraph is designed for stateful agent workflows.', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[])],
  'query': 'what is the use case of it?'},
 'query': {'history': [HumanMessage(content='What is LangGraph?', additional_kwargs={}, response_metadata={}),
   AIMessage(content='Based on the provided context, LangGraph is designed for stateful agent workflows.', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[])],
  'query': 'what is the use case of it?'}}

In [46]:
# Create conversation context

conversation_context = RunnableParallel({
    "history": RunnableLambda(lambda x: x ["history"]), # Means: Take an input called x, and return the value stored under the key 'history'
    "query": RunnableLambda(lambda x: x ["query"]),
    "rewritten_query": query_rewriter | StrOutputParser()
})

In [47]:
"""
lambda x:x["history]
equivalent to:
def get_history(x):
return x["history"]
"""

'\nlambda x:x["history]\nequivalent to:\ndef get_history(x):\nreturn x["history"]\n'

In [48]:
conversation_context_result = conversation_context.invoke({
    "history": history,
    "query": query2
})

print(conversation_context_result)

{'history': [HumanMessage(content='What is LangGraph?', additional_kwargs={}, response_metadata={}), AIMessage(content='Based on the provided context, LangGraph is designed for stateful agent workflows.', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[])], 'query': 'what is the use case of it?', 'rewritten_query': 'What are the use cases of LangGraph?'}


In [60]:
# Get the rewritten query 

rewritten_query = conversation_context_result['rewritten_query']
rewritten_query

'What are the use cases of LangGraph?'

In [68]:
rewritten_query

'What are the use cases of LangGraph?'

In [50]:
print(type(rewritten_query))

<class 'langchain_core.messages.base.TextAccessor'>


In [ ]:
# Invoke the retriever chain

retrieved_docs = retriever_chain.invoke(rewritten_query)
print(retrieved_docs)

In [113]:
# Create the RAG chain

RAG_chain =  prompt | llm

In [114]:
RAG_response =  RAG_chain.invoke({
    "history": history,
    "context": retrieved_docs["context"],
    "query": query2
})

ServerError: 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently experiencing high demand. Spikes in demand are usually temporary. Please try again later.', 'status': 'UNAVAILABLE'}}

In [ ]:
str_RAG_response = StrOutputParser().invoke(RAG_response)
print(str_RAG_response)

Based on the provided context, LangGraph is used for stateful agent workflows.


## Advanced RAG pipeline with LCEL

workflow:

```text
history + query
        │
        ▼
conversation_context
        │
        ▼
rewritten_query
        │
        ▼
   retriever
        │
        ▼
format_documents
        │
        ▼
     context
        │
        ▼
   rag_input
        │
        ▼
history + query + context
        │
        ▼
      prompt
        │
        ▼
       LLM
        │
        ▼
     string
```

In [104]:
# Query rewriter
query_rewriter = query_rewrite_prompt | llm | StrOutputParser() | RunnableLambda (lambda x: str(x))


In [105]:
# Conversation → rewritten query
conversation_context = RunnableParallel({
    "history": RunnableLambda(lambda x: x["history"]),
    "query": RunnableLambda(lambda x: x["query"]),
    "rewritten_query": query_rewriter,
})


In [106]:
# Retrieving pipeline (rewritten query → retrieved context)

retrieval_runnable = (
    conversation_context
    | RunnableLambda(lambda x: x["rewritten_query"])
    | retriever
    | format_docs_runnable
    )

In [110]:
# RAG input (Original inputs + retrieved context)

RAG_input = RunnableParallel({
    "history": RunnableLambda(lambda x:x["history"]),
    "query": RunnableLambda(lambda x:x["query"]),
    "context": retrieval_runnable,
})

In [111]:
# Complete conversational RAG

RAG_chain = (
    RAG_input
    | prompt
    | llm
    | StrOutputParser()
)

In [112]:
response = RAG_chain.invoke({
    "history":history,
    "query": query2
})

ChatGoogleGenerativeAIError: Error calling model 'gemini-3.5-flash' (RESOURCE_EXHAUSTED): 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 20, model: gemini-3.5-flash\nPlease retry in 23.426938432s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}, {'@type': 'type.googleapis.com/google.rpc.QuotaFailure', 'violations': [{'quotaMetric': 'generativelanguage.googleapis.com/generate_content_free_tier_requests', 'quotaId': 'GenerateRequestsPerDayPerProjectPerModel-FreeT